# Tesla T4 hardware validation — GPU Performance Optimization Platform

This notebook runs the CUDA validation workflow on a **real NVIDIA Tesla T4**. It is pinned to PR commit ab9886635c7db234425aa7837bdb3f21434bb0b9 so benchmark and profiler artifacts have a reproducible source revision.

## Before running
1. Open **Runtime → Change runtime type** and select **T4 GPU** (availability depends on your Colab account/session).
2. Choose **Runtime → Run all**.
3. Keep the session connected until the final artifact ZIP downloads.

The notebook stops if the allocated GPU is not a Tesla T4 (compute capability 7.5). It does not relabel another GPU as a T4. Compute Sanitizer and Nsight Compute are attempted when present; unavailable or permission-blocked tools are reported as skipped/failed rather than passed.

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/lohit112/gpu-performance-platform.git"
BRANCH = "v2/benchmark-framework-foundation"
EXPECTED_COMMIT = "ab9886635c7db234425aa7837bdb3f21434bb0b9"
WORKDIR = Path("/content/gpu-performance-platform")
LOG_DIR = Path("/content/t4-validation-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run_logged(name, command, *, cwd=WORKDIR, timeout=1800, env=None):
    print(f"\n===== {name} =====")
    completed = subprocess.run(
        command, cwd=str(cwd), text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=timeout, env=env, check=False,
    )
    log_path = LOG_DIR / f"{name.lower().replace(' ', '_')}.log"
    log_path.write_text(completed.stdout, encoding="utf-8")
    print(completed.stdout[-12000:])
    print(f"Exit code: {completed.returncode}; log: {log_path}")
    return completed.returncode, log_path

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
clone_rc, _ = run_logged(
    "git_clone",
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)],
    cwd=Path("/content"), timeout=600,
)
if clone_rc != 0:
    raise RuntimeError("Could not clone the PR branch.")
fetch_rc, _ = run_logged(
    "git_fetch_pinned_commit",
    ["git", "fetch", "--depth=1", "origin", EXPECTED_COMMIT],
)
if fetch_rc != 0:
    raise RuntimeError("Could not fetch the pinned PR commit.")
checkout_rc, _ = run_logged(
    "git_checkout_pinned_commit", ["git", "checkout", "--detach", EXPECTED_COMMIT]
)
if checkout_rc != 0:
    raise RuntimeError("Could not check out the pinned PR commit.")
actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True
).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)
print("Pinned source revision:", actual_commit)

## 1. Verify the actual GPU and CUDA toolchain

This is a strict T4 gate. If Colab assigns an L4, A100, CPU, or another device, stop and select a T4 runtime before continuing.

In [ ]:
os.environ["MPLBACKEND"] = "Agg"
cuda_bin = Path("/usr/local/cuda/bin")
if cuda_bin.is_dir():
    os.environ["PATH"] = f"{cuda_bin}:{os.environ.get('PATH', '')}"

smi_path = shutil.which("nvidia-smi")
if not smi_path:
    raise RuntimeError("nvidia-smi is unavailable: this runtime has no visible NVIDIA GPU.")
smi = subprocess.check_output(
    [smi_path, "--query-gpu=name,driver_version", "--format=csv,noheader"], text=True
).strip()
print(smi)

try:
    import torch
except ImportError as exc:
    raise RuntimeError("PyTorch is needed only to query the CUDA device capability.") from exc
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access CUDA in this runtime.")
device_name = torch.cuda.get_device_name(0)
compute_capability = torch.cuda.get_device_capability(0)
print("CUDA device:", device_name)
print("Compute capability:", compute_capability)
if "T4" not in device_name.upper() or compute_capability != (7, 5):
    raise RuntimeError(
        f"T4 required (name containing T4, CC 7.5); got {device_name!r}, {compute_capability}."
    )

nvcc = shutil.which("nvcc")
if not nvcc:
    raise RuntimeError("nvcc is unavailable. Enable a CUDA Toolkit runtime before continuing.")
nvcc_version = subprocess.check_output([nvcc, "--version"], text=True)
print(nvcc_version)
compute_sanitizer = shutil.which("compute-sanitizer")
ncu = shutil.which("ncu")
print("compute-sanitizer:", compute_sanitizer or "NOT INSTALLED — sanitizer stage will be marked skipped")
print("ncu:", ncu or "NOT INSTALLED — Nsight stage will be marked skipped")

environment = {
    "gpu_name": device_name,
    "compute_capability": list(compute_capability),
    "nvidia_smi": smi,
    "nvcc_path": nvcc,
    "nvcc_version": nvcc_version,
    "compute_sanitizer_path": compute_sanitizer,
    "ncu_path": ncu,
    "source_commit": actual_commit,
}
(WORKDIR / "results").mkdir(parents=True, exist_ok=True)
(WORKDIR / "results" / "t4_environment.json").write_text(
    json.dumps(environment, indent=2) + "\n", encoding="utf-8"
)

## 2. Run the CPU-side regression suite

These tests validate the Python benchmark, provenance, NCU parsing, and report code. They are separate from the GPU checks below.

In [ ]:
missing = [pkg for pkg in ("pytest", "numpy", "matplotlib") if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
test_rc, _ = run_logged("python_unit_tests", [sys.executable, "-m", "pytest", "-q"])
if test_rc != 0:
    raise RuntimeError("Python regression suite failed; inspect the saved test log.")

## 3. Compile the partial-tile binary and run Compute Sanitizer

The N=80 case exercises partial 32/64-wide CUDA tiles while remaining divisible by 16 for WMMA. If Compute Sanitizer is missing, this notebook records the stage as skipped and still proceeds to the benchmark; that is **not** a sanitizer pass.

In [ ]:
build_dir = WORKDIR / "build"
profile_dir = WORKDIR / "results" / "profiling"
benchmark_dir = WORKDIR / "results" / "benchmarks"
build_dir.mkdir(parents=True, exist_ok=True)
profile_dir.mkdir(parents=True, exist_ok=True)
benchmark_dir.mkdir(parents=True, exist_ok=True)
sanitizer_status = {}
sanitizer_binary = build_dir / "matmul_sanitizer_t4"
compile_cmd = [
    nvcc, "-O1", "-g", "-lineinfo", "-arch=sm_75", "-DN=80",
    str(WORKDIR / "cuda_kernels" / "matmul_suite.cu"), "-lcublas",
    "-o", str(sanitizer_binary),
]
san_compile_rc, _ = run_logged("compile_partial_tile_binary", compile_cmd, timeout=900)
sanitizer_status["compile_N80"] = "passed" if san_compile_rc == 0 else "failed"
if san_compile_rc == 0 and compute_sanitizer:
    for tool in ("memcheck", "racecheck", "synccheck"):
        rc, _ = run_logged(
            f"compute_sanitizer_{tool}",
            [compute_sanitizer, "--tool", tool, "--error-exitcode", "1", str(sanitizer_binary)],
            timeout=1800,
        )
        sanitizer_status[tool] = "passed" if rc == 0 else "failed"
elif san_compile_rc != 0:
    sanitizer_status.update({tool: "not_run_compile_failed" for tool in ("memcheck", "racecheck", "synccheck")})
else:
    sanitizer_status.update({tool: "skipped_tool_unavailable" for tool in ("memcheck", "racecheck", "synccheck")})
print(json.dumps(sanitizer_status, indent=2))
(WORKDIR / "results" / "compute_sanitizer_status.json").write_text(
    json.dumps(sanitizer_status, indent=2) + "\n", encoding="utf-8"
)

## 4. Run the real CUDA GEMM benchmark

Runs K0–K4 on the T4 at N=4096, with five warmups and twenty timed iterations. The runner performs cuBLAS correctness checks before accepting timings and records the executable fingerprint.

In [ ]:
benchmark_json = benchmark_dir / "t4_n4096.json"
benchmark_rc, benchmark_log = run_logged(
    "cuda_benchmark_N4096",
    [sys.executable, "-m", "benchmark.runner", "--arch", "sm_75",
     "--size", "4096", "--warmup", "5", "--iterations", "20",
     "--output", str(benchmark_json)],
    timeout=1800,
)
if benchmark_rc != 0 or not benchmark_json.is_file():
    raise RuntimeError("The CUDA T4 benchmark did not complete successfully.")
benchmark_artifact = json.loads(benchmark_json.read_text(encoding="utf-8"))
print("GPU:", benchmark_artifact["metadata"].get("gpu_name"))
print("Binary SHA-256:", benchmark_artifact["metadata"].get("binary_sha256"))
print("Reference GFLOP/s:", benchmark_artifact["reference"]["gflops"])
for row in benchmark_artifact["results"]:
    validation = row.get("validation", {}).get("passed")
    print(f"{row['kernel_id']} {row['name']}: {row['latency_ms']:.4f} ms, "
          f"{row['gflops']:.2f} GFLOP/s, correctness={validation}")

## 5. Capture real Nsight Compute counters

The capture is tied to results/benchmarks/t4_n4096.json and the exact build/matmul_bench executable. If NCU is installed but profiler-counter access is restricted by the runtime, the failure is saved and reported.

In [ ]:
ncu_status = "skipped_tool_unavailable"
ncu_csv = None
ncu_sidecar = None
if ncu:
    ncu_rc, ncu_log = run_logged(
        "nsight_compute_capture",
        ["bash", "scripts/profile_ncu.sh", "build/matmul_bench",
         "results/profiling", "matmul_.*", "full", "results/benchmarks/t4_n4096.json"],
        timeout=2400,
    )
    ncu_status = "passed" if ncu_rc == 0 else "failed_or_counter_access_blocked"
    if ncu_rc == 0:
        csv_candidates = sorted(profile_dir.glob("*_profile.csv"), key=lambda p: p.stat().st_mtime)
        if csv_candidates:
            ncu_csv = csv_candidates[-1]
            ncu_sidecar = ncu_csv.with_suffix(".meta.json")
            print("NCU CSV:", ncu_csv)
            print("NCU sidecar:", ncu_sidecar)
            if ncu_sidecar.is_file():
                ncu_meta = json.loads(ncu_sidecar.read_text(encoding="utf-8"))
                print("NCU provenance verified:", ncu_meta.get("validation", {}).get("verified"))
        else:
            ncu_status = "failed_csv_missing"
else:
    print("NCU is not installed in this runtime; profiler counters were not collected.")
(WORKDIR / "results" / "ncu_status.json").write_text(
    json.dumps({"status": ncu_status, "csv": str(ncu_csv) if ncu_csv else None}, indent=2) + "\n",
    encoding="utf-8",
)

## 6. Generate the measured roofline report (only for a verified NCU capture)

The report independently validates the sidecar against the benchmark JSON, binary hash, GPU, shape, timing protocol, precision manifest, and selected CSV. If one check fails, no empirical roofline chart is emitted.

In [ ]:
empirical_roofline_status = "unavailable_no_ncu_csv"
if ncu_csv and ncu_csv.is_file():
    sys.path.insert(0, str(WORKDIR))
    from report_gen.report_generator import ReportGenerator
    generator = ReportGenerator(
        output_dir=str(WORKDIR / "results" / "t4_report"),
        benchmark_json=str(benchmark_json),
        nsight_csv=str(ncu_csv),
    )
    generator.run_nsight_analysis()
    generator.run_kernel_benchmarks()
    generator.run_empirical_roofline_analysis()
    empirical_roofline_status = (
        "generated" if generator.empirical_roofline_chart else generator.empirical_roofline_note
    )
    print("Empirical roofline:", empirical_roofline_status)
    if generator.empirical_roofline_chart:
        print("Chart:", generator.empirical_roofline_chart)
else:
    print("No NCU CSV available; empirical roofline is intentionally unavailable.")

## 7. Package the evidence

The ZIP includes the benchmark JSON, compiled CUDA binary, sanitizer logs, NCU report/CSV/sidecar (when produced), report outputs, environment details, and exact source commit. Download it and attach it back in the project chat so the measurements can be checked and the report/PR updated.

In [ ]:
summary = {
    "source_commit": actual_commit,
    "gpu_name": device_name,
    "compute_capability": list(compute_capability),
    "benchmark": "passed" if benchmark_json.is_file() else "failed_or_missing",
    "sanitizer": sanitizer_status,
    "ncu": ncu_status,
    "empirical_roofline": empirical_roofline_status,
    "benchmark_json": str(benchmark_json),
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(WORKDIR / "results" / "t4_validation_summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
bundle = Path("/content/t4_validation_artifacts.zip")
include_roots = [WORKDIR / "results", WORKDIR / "build" / "matmul_bench", LOG_DIR]
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for root in include_roots:
        if root.is_file():
            arcname = root.relative_to(WORKDIR) if WORKDIR in root.parents else Path("build/matmul_bench")
            archive.write(root, arcname=str(arcname))
        elif root.exists():
            for path in root.rglob("*"):
                if path.is_file():
                    try:
                        arcname = path.relative_to(WORKDIR)
                    except ValueError:
                        arcname = Path("logs") / path.relative_to(LOG_DIR)
                    archive.write(path, arcname=str(arcname))
print(json.dumps(summary, indent=2))
print("Artifact bundle:", bundle, f"({bundle.stat().st_size / (1024**2):.1f} MiB)")
try:
    from google.colab import files
    files.download(str(bundle))
except Exception as exc:
    print("Automatic download unavailable; retrieve this file manually:", bundle, exc)